Notebook for the supplementary experiment comparing 0-100 and 0-10 rating scales

In [1]:
import pandas as pd

Read the data:

In [ ]:
df_rs = pd.read_csv('llm_rs_scores.csv')
df_rs10 = pd.read_csv('llm_rs10_scores.csv')

# normalize:
df_rs['score'] = df_rs['score'] / 100
df_rs10['score'] = df_rs10['score'] / 10

# human mean scores:
df_human = pd.read_csv('human_scores.csv')
human_rs = df_human[df_human['method'] == 'rs']
df_humanagg_rs = human_rs.groupby(["tweet_id", "method", "dimension"], as_index=False).agg(
        human_mean=("score", "mean"),
)

print(df_rs.head())
print(df_rs10.head())
print(df_humanagg_rs.head())

                 tweet_id  dimension method model  score
0  ['901760751152623616']    valence     rs   gpt   0.65
1  ['901760751152623616']    arousal     rs   gpt   0.58
2  ['901760751152623616']  dominance     rs   gpt   0.52
3  ['863364746577313792']    valence     rs   gpt   0.91
4  ['863364746577313792']    arousal     rs   gpt   0.87
                 tweet_id  dimension method model  score
0  ['901760751152623616']    valence   rs10   gpt    0.7
1  ['901760751152623616']    arousal   rs10   gpt    0.6
2  ['901760751152623616']  dominance   rs10   gpt    0.5
3  ['863364746577313792']    valence   rs10   gpt    0.9
4  ['863364746577313792']    arousal   rs10   gpt    0.8
                 tweet_id method  dimension  human_mean
0  ['815557023366516736']     rs    arousal    0.741667
1  ['815557023366516736']     rs  dominance    0.590000
2  ['815557023366516736']     rs    valence    0.580000
3  ['816343164676083714']     rs    arousal    0.700000
4  ['816343164676083714']     rs  do

Q1: To what extent does a coarser scale affect LLM judgment -- Do scores using scales of different granularity still correlate, or are there fundamental differences? 
- use Pearson's r, Spearman's rho to measure correlation between normalized 0-100 and 0-10 ratings.

In [6]:
from scipy.stats import pearsonr, spearmanr

models = ['gpt', 'mini', 'nano']
dimension_order = ["valence", "arousal", "dominance"]

rs_rs10 = df_rs.merge(
    df_rs10,
    on=["tweet_id", "dimension", "model"],
    suffixes=("_rs", "_rs10")
)

rows = []

for (model, dimension), group in rs_rs10.groupby(["model", "dimension"]):
    x = group["score_rs"]
    y = group["score_rs10"]

    pearson_r, pearson_p = pearsonr(x, y)
    spearman_rho, spearman_p = spearmanr(x, y)

    rows.append({
        "model": model,
        "dimension": dimension,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
    })

granularity_corr_df = pd.DataFrame(rows)

granularity_corr_df["dimension"] = pd.Categorical(
    granularity_corr_df["dimension"],
    categories=dimension_order,
    ordered=True
)

granularity_corr_df = granularity_corr_df.sort_values(
    ["model", "dimension"]
).reset_index(drop=True)

granularity_corr_df.round(3)


,model,dimension,pearson_r,pearson_p,spearman_rho,spearman_p
0,gpt,valence,0.964,0.0,0.965,0.0
1,gpt,arousal,0.922,0.0,0.915,0.0
2,gpt,dominance,0.929,0.0,0.887,0.0
3,mini,valence,0.944,0.0,0.961,0.0
4,mini,arousal,0.855,0.0,0.837,0.0
5,mini,dominance,0.818,0.0,0.764,0.0
6,nano,valence,0.964,0.0,0.962,0.0
7,nano,arousal,0.875,0.0,0.880,0.0
8,nano,dominance,0.722,0.0,0.651,0.0


Q2: How does using a coarser scale affect alignment with human annotations?